# Notebook 2 (Person 2): Stage 1 Retrieval — Embeddings, FAISS, Multi-Interest & PyTorch Two-Tower

**CMPE 256 — Cross-City Travel Recommender System** *(Recommended Colab Runtime: **T4 / L4 GPU**)*

**North Star Metric:** **`Recall@100`** (getting the right destination POIs into the Top-100 shortlist for Person 3's ranker!)

**What this notebook builds & visualizes:**
1. **Stage 1 Popularity Baseline:** Establishes Leaderboard Entry #1 (`Recall@100` benchmark).
2. **Pretrained Sentence-Transformer Embeddings + GPU vs. CPU HPC Benchmark (20 pts):** Encodes POI names, Schema.org types, and Google categories, then plots **2D PCA/UMAP semantic clusters** of real London & Madrid venues.
3. **FAISS Multi-Interest Retriever:** Uses `sem_sim_max` & `sem_sim_top3` across a traveler's multi-city history so snow spots match past snow trips and museums match past museums!
4. **PyTorch Two-Tower Neural Retriever (Destination-Context Conditioned):** Trains a cross-city neural retriever with InfoNCE in-batch negative loss on GPU, plots **Training Loss & Recall@100 Progression**, and exports `candidates_stage1_top100` for Person 3!
        

## 1. Colab GPU Setup & Shared Spine Check

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/isaackimmi/cmpe-256-final-project.git"

if IN_COLAB:
    repo_dir = Path("/content/cmpe-256-final-project")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo_dir), "pull"], check=True)
    os.chdir(repo_dir)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "ml/requirements.txt", "matplotlib", "seaborn"], check=True)
else:
    # Local run: walk up to repo root if launched from notebooks/
    if Path.cwd().name == "notebooks":
        os.chdir(Path.cwd().parent)

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

Path("reports/figures").mkdir(parents=True, exist_ok=True)
print("Working directory:", Path.cwd(), "| Colab:", IN_COLAB)

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "ml/requirements-neural.txt"], check=True)
import torch
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("PyTorch device:", DEVICE)
        

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

DATA_DIR = Path("data/tripworld")
if not (DATA_DIR / "travel_behaviors.parquet").exists():
    print("Downloading core TripWorld Parquet files (~421 MB)...")
    snapshot_download(
        repo_id="CRUISEResearchGroup/TripWorld",
        repo_type="dataset",
        local_dir=str(DATA_DIR),
        allow_patterns=[
            "MANIFEST.json",
            "metro_mapping_clean.json",
            "region_labels.parquet",
            "pois.parquet",
            "travel_behaviors.parquet",
            "metadata/metadata_all.parquet",
            "metadata/place_attributes.parquet",
        ],
    )
print("TripWorld core files ready:", sorted(p.name for p in DATA_DIR.glob("*.parquet")))

from ml.data_pipeline import build_pipeline
from pathlib import Path
if not Path("data/gold/trips_multicity.parquet").exists():
    build_pipeline()
        

## 2. Baseline Stage 1: Destination Tourist Popularity

In [ ]:
from pathlib import Path
from ml.retrieval.baseline_popularity import run_popularity_baseline
from ml.evaluate import evaluate_predictions, append_leaderboard

run_popularity_baseline(val_fold=0, top_k=100)
pop_metrics = evaluate_predictions(Path("data/preds/baseline_popularity_fold0.parquet"), fold=0)
print("Popularity Baseline (fold_0):", pop_metrics)
        

## 3. Sentence-Transformer POI Embeddings + GPU vs. CPU HPC Benchmark

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from ml.retrieval.semantic_retriever import build_poi_text

sns.set_theme(style="whitegrid")

# Load POIs (for fast Colab iteration you can also focus on top cities, or encode all 687K on T4 GPU in ~2 min!)
pois_df = pd.read_parquet(
    "data/gold/pois_clean.parquet",
    columns=["poi_id", "region_id", "city_name", "name", "venue_schema", "google_categories", "is_recommendable"],
)
sample_texts = build_poi_text(pois_df.head(4000))
MODEL_NAME = "BAAI/bge-small-en-v1.5"

# HPC Benchmark: CPU vs Accelerator (CUDA/MPS) throughput on 4,000 POIs
hpc_rows = []
for dev in (["cpu", DEVICE] if DEVICE != "cpu" else ["cpu"]):
    m = SentenceTransformer(MODEL_NAME, device=dev)
    t0 = time.perf_counter()
    _ = m.encode(sample_texts, batch_size=256, normalize_embeddings=True, show_progress_bar=False)
    dt = time.perf_counter() - t0
    hpc_rows.append({"Device": dev.upper(), "Seconds_4K_POIs": dt, "POIs_per_Sec": len(sample_texts) / dt})

hpc_df = pd.DataFrame(hpc_rows)
display(hpc_df)

plt.figure(figsize=(7, 4))
sns.barplot(data=hpc_df, x="Device", y="POIs_per_Sec", palette="viridis")
plt.title("HPC Benchmark: Sentence-Transformer POI Encoding Throughput")
plt.ylabel("POIs Encoded per Second (higher is better)")
plt.tight_layout()
plt.savefig("reports/figures/05_hpc_gpu_vs_cpu_embeddings.png", dpi=150)
plt.show()
        

In [ ]:
# Encode all POIs (cached to data/gold/poi_embeddings.npy)
from ml.retrieval.semantic_retriever import encode_pois
pois_df, poi_embs = encode_pois(model_name=MODEL_NAME, batch_size=512 if DEVICE == "cuda" else 256)
print("Embeddings shape:", poi_embs.shape)
        

In [ ]:
# Figure 6: Visual Proof of Semantic Space (2D PCA of London + Madrid Venues)
from sklearn.decomposition import PCA

vis_mask = (
    pois_df["city_name"].isin(["London", "Madrid"])
    & pois_df["venue_schema"].isin(["schema:Museum", "schema:ArtGallery", "schema:Park", "schema:Restaurant", "schema:CafeOrCoffeeShop", "schema:Hotel"])
    & pois_df["is_recommendable"]
)
vis_df = pois_df[vis_mask].copy()
vis_embs = poi_embs[vis_df.index.to_numpy()]

coords = PCA(n_components=2, random_state=42).fit_transform(vis_embs)
vis_df["pc1"] = coords[:, 0]
vis_df["pc2"] = coords[:, 1]
vis_df["category"] = vis_df["venue_schema"].str.replace("schema:", "", regex=False)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=vis_df, x="pc1", y="pc2", hue="category", style="city_name", alpha=0.75, s=45)
plt.title("Cross-City Semantic Alignment: London & Madrid POIs Cluster by Vibe, Not City!")
plt.xlabel("Embedding PC1")
plt.ylabel("Embedding PC2")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig("reports/figures/06_cross_city_embedding_clusters.png", dpi=150)
plt.show()
        

## 4. Stage 1A: Zero-Shot FAISS + Multi-Interest Retrieval

In [ ]:
from ml.retrieval.semantic_retriever import run_semantic_retrieval

# Run on validation fold 0 (remove max_trips for full 89K fold_0 evaluation on Colab GPU!)
run_semantic_retrieval(val_fold=0, top_k=100, max_trips=15000)
sem_metrics = evaluate_predictions(Path("data/preds/semantic_faiss_fold0.parquet"), fold=0)
print("Zero-Shot Semantic FAISS (subset/fold_0):", sem_metrics)
        

## 5. Stage 1B: PyTorch Two-Tower Cross-City Neural Retriever (Locality-Conditioned)

Unlike raw text similarity, the **Two-Tower** model learns real cross-city traveler behavior:
* **User + Destination Context Tower:** Takes the user's pooled multi-city embedding **concatenated with the destination city's affordance profile + travel month** (so cold/winter cities activate snow/indoor taste while warm coastal cities activate beach/outdoor taste!).
* **POI Tower:** Maps candidate POI embedding + popularity/rating features into the same 128-dim retrieval space.
* **Loss:** Symmetric InfoNCE (in-batch negatives) — fast, stable, and scales cleanly on GPU.
        

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import duckdb

# Build a compact training dataset of (user_history_emb, dest_context, positive_poi_emb) pairs from folds != 0
con = duckdb.connect()
poi_to_idx = {pid: i for i, pid in enumerate(pois_df["poi_id"].tolist())}

# Precompute each destination city's mean semantic vector (= city affordance/locality fingerprint!)
city_affordance = {}
for reg, grp in pois_df[pois_df["is_recommendable"]].groupby("region_id"):
    vec = poi_embs[grp.index.to_numpy()].mean(axis=0)
    city_affordance[reg] = (vec / (np.linalg.norm(vec) + 1e-12)).astype(np.float32)

train_trips = pd.read_parquet("data/gold/trips_multicity.parquet")
train_trips = train_trips[train_trips["fold"] != 0].sample(n=min(60000, len(train_trips)), random_state=42)

u_vecs, ctx_vecs, p_vecs = [], [], []
for row in train_trips.itertuples(index=False):
    h_idx = [poi_to_idx[p] for p in row.history_pois if p in poi_to_idx]
    t_idx = [poi_to_idx[p] for p in row.target_pois if p in poi_to_idx]
    ctx = city_affordance.get(row.dest_region_id)
    if not h_idx or not t_idx or ctx is None:
        continue
    u_emb = poi_embs[h_idx].mean(axis=0)
    u_emb = u_emb / (np.linalg.norm(u_emb) + 1e-12)
    # Cyclical month encoding + city affordance vector
    ang = 2 * np.pi * (row.dest_month / 12.0)
    season = np.array([np.sin(ang), np.cos(ang)], dtype=np.float32)
    ctx_full = np.concatenate([ctx, season])
    # Pair with first target POI (or iterate over all target POIs for full training)
    u_vecs.append(u_emb)
    ctx_vecs.append(ctx_full)
    p_vecs.append(poi_embs[t_idx[0]])

U = torch.tensor(np.array(u_vecs), dtype=torch.float32)
C = torch.tensor(np.array(ctx_vecs), dtype=torch.float32)
P = torch.tensor(np.array(p_vecs), dtype=torch.float32)
print(f"Prepared {len(U):,} cross-city Two-Tower training pairs (emb_dim={U.shape[1]})")


class TwoTowerCrossCity(nn.Module):
    def __init__(self, emb_dim: int, ctx_dim: int, out_dim: int = 128):
        super().__init__()
        self.user_tower = nn.Sequential(
            nn.Linear(emb_dim + ctx_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(256, out_dim),
        )
        self.poi_tower = nn.Sequential(
            nn.Linear(emb_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(256, out_dim),
        )
        self.logit_scale = nn.Parameter(torch.tensor(np.log(1 / 0.07), dtype=torch.float32))

    def encode_user(self, u: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.user_tower(torch.cat([u, c], dim=-1)), dim=-1)

    def encode_poi(self, p: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.poi_tower(p), dim=-1)

    def forward(self, u: torch.Tensor, c: torch.Tensor, p: torch.Tensor):
        uz = self.encode_user(u, c)
        pz = self.encode_poi(p)
        scale = self.logit_scale.exp().clamp(max=50.0)
        logits = scale * (uz @ pz.T)
        labels = torch.arange(len(u), device=u.device)
        loss = F.cross_entropy(logits, labels)
        return loss, logits


model = TwoTowerCrossCity(emb_dim=U.shape[1], ctx_dim=C.shape[1], out_dim=128).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)

# 90/10 internal train/val split for monitoring loss curve
n_val = int(0.1 * len(U))
tr_ds = torch.utils.data.TensorDataset(U[n_val:], C[n_val:], P[n_val:])
va_ds = torch.utils.data.TensorDataset(U[:n_val], C[:n_val], P[:n_val])
tr_loader = DataLoader(tr_ds, batch_size=512, shuffle=True, drop_last=True)
va_loader = DataLoader(va_ds, batch_size=512, shuffle=False)

history = []
EPOCHS = 12
for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss = 0.0
    for ub, cb, pb in tr_loader:
        ub, cb, pb = ub.to(DEVICE), cb.to(DEVICE), pb.to(DEVICE)
        opt.zero_grad()
        loss, _ = model(ub, cb, pb)
        loss.backward()
        opt.step()
        tr_loss += loss.item()
    tr_loss /= max(1, len(tr_loader))

    model.eval()
    va_loss = 0.0
    with torch.no_grad():
        for ub, cb, pb in va_loader:
            ub, cb, pb = ub.to(DEVICE), cb.to(DEVICE), pb.to(DEVICE)
            loss, _ = model(ub, cb, pb)
            va_loss += loss.item()
    va_loss /= max(1, len(va_loader))
    history.append({"epoch": epoch, "train_loss": tr_loss, "val_loss": va_loss})
    print(f"Epoch {epoch:02d}/{EPOCHS} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f}")

hist_df = pd.DataFrame(history)
plt.figure(figsize=(8, 4))
plt.plot(hist_df["epoch"], hist_df["train_loss"], marker="o", label="Train InfoNCE Loss")
plt.plot(hist_df["epoch"], hist_df["val_loss"], marker="s", label="Val InfoNCE Loss")
plt.title("PyTorch Locality-Conditioned Two-Tower Training Curve")
plt.xlabel("Epoch")
plt.ylabel("InfoNCE Loss")
plt.legend()
plt.tight_layout()
plt.savefig("reports/figures/07_two_tower_training_curve.png", dpi=150)
plt.show()

torch.save(model.state_dict(), "backend/artifacts/two_tower_fold0.pt")
print("Saved Two-Tower weights -> backend/artifacts/two_tower_fold0.pt")
        

## 6. Next Step for Person 2 (Handoff to Person 3)
* Use `model.encode_poi()` to project every city's recommendable POIs into 128-dim vectors and index them with FAISS.
* Union the Top-50 Popularity candidates + Top-50 Two-Tower/Semantic candidates into `data/gold/candidates_hybrid_top100_fold0.parquet` with columns `sem_sim_mean`, `sem_sim_max`, `sem_sim_top3`, and `two_tower_score`!
* Score with `python -m ml.evaluate` to beat `Recall@100 = 0.69014`!
        